In [ ]:
############## Code to fetch all the unique ratings from all the tabs

import pandas as pd

# Load the Excel file
file_path = "correct_rating_df_grouped_by_fund.xlsx"  # change to your file path
xls = pd.ExcelFile(file_path)

all_ratings = set()

# Loop through all sheets
for sheet in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    if "Ratings" in df.columns:
        # Drop NaN, strip spaces, and update the set
        ratings = df["Ratings"].dropna().astype(str).str.strip()
        all_ratings.update(ratings)

# Convert set to sorted list
unique_ratings = sorted(all_ratings)

print("Unique Ratings:")
for r in unique_ratings:
    print(r)

# If you want to save them to Excel
pd.DataFrame({"Unique Ratings": unique_ratings}).to_excel("unique_ratings.xlsx", index=False)


Unique Ratings:
AAA(SO)
CRISIL A1+
CRISIL A1l+
CRISIL AA
CRISIL AA+
CRISIL AA-
CRISIL AAA
CRISIL AAA(SO)
ICRA A1+
ICRA AA
ICRA AA+
ICRA AAA
IND A1+
IND AA+
IND AAA
IND AAA(SO)
SOV
SOVRN SOV
a


In [4]:
############## Code to fetch all the unique category from all the tabs

import pandas as pd

# Load the Excel file
file_path = "correct_rating_df_grouped_by_fund.xlsx"  # change to your file path
xls = pd.ExcelFile(file_path)

all_ratings = set()

# Loop through all sheets
for sheet in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    if "Portfolio" in df.columns:
        # Drop NaN, strip spaces, and update the set
        ratings = df["Portfolio"].dropna().astype(str).str.strip()
        all_ratings.update(ratings)

# Convert set to sorted list
unique_ratings = sorted(all_ratings)

print("Unique Portfolio:")
for r in unique_ratings:
    print(r)

# If you want to save them to Excel
pd.DataFrame({"Unique Portfolio": unique_ratings}).to_excel("unique_Portfolio.xlsx", index=False)


Unique Portfolio:
05.70 % Nabard
05.78 % Chennai Petroleum Corporation Ltd.
05.78 % LIC Housing Finance Ltd.
05.81 % Rec Ltd.
05.85 % Rec Ltd.
05.94 % Rec Ltd.
06.07 % Nabard
06.23 % Rec Ltd.
06.40 % Jamnagar Utilities & Power Pvt. Ltd.
(Mukesh Ambani Group)
06.40 % Jamnagar Utilities & Power Pvt. Ltd.(Mukesh ec apn ti‘(‘ésC QTC ER
Ambani Group)
06.40 % LIC Housing Finance Ltd.
06.47 % Indian Railways
06.50 % Power Finance Corporation
06.55 % ICICI Home Finance Co.Ltd.
06.59 % Power Finance Corporation
06.60 % Rec Ltd.
06.65 % Indian Railways
Finance Corporation Ltd.
06.75 % Sikka Ports And Terminals Ltd.
(Mukesh Ambani Group)
06.75 % Sikka Ports And Terminals Ltd. (Mukesh
Ambani Group)
06.90 % Housing & Urban
Develonment Corpnoration Ltd.
06.92 % Indian Railways Finance Corporation Ltd.
06.99 % Sundaram Fin Ltd.
07 60 % Poonawalla Fincorn | td.
07.07 % LIC Housing Finance Ltd.
07.10 % Exim
07.11 % Small Indust Deviop Bank Of India
07.12 % Tata Capital Housing Finance Ltd.
07.13 % Nhpc

In [ ]:
####################### Testing Code to fetch text from screenshot after zooom and save it in excel file with fund distributed as tabs######################
import cv2
import numpy as np
import pytesseract
from PIL import Image
import os
import re

rating_set = set()
# Set Tesseract path (update for your system)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
pan = 'AAATT0570A'
mutual_fund = "TATA MUTUAL FUND"
input_dir = "screenshots_cropped_at_rectangles"
pattern = re.compile(r"(AAA\(SO\)|AAA\(CE\)|AA\(CE\)|A1\+|AA\+|AA-|AAA|AA|SOVRN SOV|SOVRN|SOV|Sovereign)")
agencies = ["CARE", "IND", "CRISIL", "ICRA", "FITCH", "India Ratings", "Brickwork", "Acuite"]


agency_pattern = re.compile(rf"({'|'.join(agencies)})")


def normalize_rating_text(text):
    replacements = {
        "＋": "+",  # fullwidth plus
        "–": "-",  # en dash
        "−": "-",  # minus
        "﹣": "-",  # small hyphen
        "‒": "-",  # figure dash
        "―": "-",  # horizontal bar
        "（": "(",  # fullwidth left parenthesis
        "）": ")",  # fullwidth right parenthesis
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove zero-width and non-breaking spaces
    text = re.sub(r"[\u200B-\u200D\uFEFF\u00A0]", "", text)
    return text

def agency_matching(normalized_text, agency=''):
    agency_match = agency_pattern.search(normalized_text)
    if agency_match:
        agency = agency_match.group(1)
        print("agency:", agency)
        normalized_text = normalized_text.replace(agency, " ")
        print("agency_replaced_normalised_text :", normalized_text)
    return [normalized_text, agency]

def rating_matching(normalized_text, rating=''):
    match = pattern.search(normalized_text)
    # print(f"Line: {normalized_text}")
    if match:
        rating = match.group(0)
        print(f"Rating: {rating}\n")
        print("rating_replaced_normalised_text :", normalized_text)
        normalized_text = normalized_text.replace(rating, " ")
    return [normalized_text, rating]

def AUM_matching(normalized_text, total_AUM=''):
    match_AUM = re.findall(r"\d+\.\d+", normalized_text)

    # If found, return the last one (but not if it's at start)
    if match_AUM:
        last = match_AUM[-1]
        # check position of last match
        m = re.finditer(r"\d+\.\d+", normalized_text)
        last_match = list(m)[-1]
        if last_match.start() == 0:
            total_AUM = None  # ignore if at start
        else:
            total_AUM = last
            # total_AUM = match_AUM[-1]
            print(f"Total AUM: {total_AUM}\n")
            print("AUM_replaced_normalised_text :", normalized_text)
            normalized_text = normalized_text.replace(total_AUM, "")
    return [normalized_text, total_AUM]

def MV_matching(normalized_text, market_value=''):
    match_MV = re.findall(r"\d+\.\d+", normalized_text)

    # If found, return the last one (but not if it's at start)
    if match_MV:
        last = match_MV[-1]
        # check position of last match
        m_mv = re.finditer(r"\d+\.\d+", normalized_text)
        last_match_mv = list(m_mv)[-1]
        if last_match_mv.start() == 0:
            market_value = None  # ignore if at start
        else:
            market_value = last
            # total_AUM = match_MV[-1]
            print(f"MV: {market_value}\n")
            print("MV_replaced_normalised_text :", normalized_text)
            normalized_text = normalized_text.replace(market_value, "")
    
    return [normalized_text, market_value]




import pandas as pd
correct_rating_df = pd.DataFrame(columns=['Pan', 'ISIN No.', 'Mutual Fund', 'Fund Name','Fund Type', 'Portfolio', "Agency", 'Ratings', "Category", '(%) Of Total AUM', "Factsheet Date", "URL", 'Filename'])
remaining_rating_df = pd.DataFrame(columns=['Text'])
for filename in os.listdir(input_dir):
    print("---------------------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>--------------------------------------------------------------")
    if filename.endswith(".png"):
        image_path = os.path.join(input_dir, filename)
        print(image_path)
        image = cv2.imread(image_path)

        # Scale image (e.g., 1.5x zoom)
        # scale_percent = 120  # Adjust this value as needed (e.g., 150 means 150% size)
        scale_percent = 200
        width = int(image.shape[1] * scale_percent / 100)
        height = int(image.shape[0] * scale_percent / 100)
        dim = (width, height)

        # Resize image
        image = cv2.resize(image, dim, interpolation=cv2.INTER_LINEAR)

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Threshold to isolate dark lines (black separators)
        # _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        # Adaptive threshold to handle gray lines and varying lighting
        thresh = cv2.adaptiveThreshold(
            gray, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            blockSize=15,
            C=10
        )

        # Detect horizontal lines
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
        detected_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)

        # Find contours of the lines
        contours, _ = cv2.findContours(detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Get y-coordinates of all horizontal lines
        line_y_coords = sorted([cv2.boundingRect(cnt)[1] for cnt in contours])

        # Add image boundaries
        line_y_coords = [0] + line_y_coords + [image.shape[0]]

        # Process and print each row
        print("\nExtracted Table Rows:")
        for i in range(len(line_y_coords) - 1):
            y_start = line_y_coords[i]
            y_end = line_y_coords[i + 1]
            
            # Skip very small regions (likely just the line itself)
            if y_end - y_start < 10:
                continue

            # Extract the row region
            row_img = image[y_start:y_end, :]

            # OPTIONAL: Resize the row to improve OCR accuracy
            zoom_factor = 2.0
            row_img = cv2.resize(row_img, None, fx=zoom_factor, fy=zoom_factor, interpolation=cv2.INTER_LINEAR)

            # Convert to PIL Image for Tesseract
            row_pil = Image.fromarray(cv2.cvtColor(row_img, cv2.COLOR_BGR2RGB))

            # Tesseract OCR
            custom_config = r'--oem 3 --psm 6 -c preserve_interword_spaces=1'
            text = pytesseract.image_to_string(row_pil, config=custom_config).strip()
            text = text.replace("Al1", "A1")
            normalized_text = normalize_rating_text(text)
            agency = ""
            rating = ""
            portfolio = ""
            agency_2 = ''
            rating_2 = ''
            portfolio_2 = ''
            total_AUM = ''

            if text:
                # print(f"\nRow {i+1}:")
                # print(text)  # Print full text block (don't split by new lines)
                # print(len(re.split(r'\s{2,}', text)), re.split(r'\s{2,}', text))
                if len(re.split(r'\s{2,}', text)) ==4 :
                    # print(text)
                    # print(len(re.split(r'\s{2,}', text)), re.split(r'\s{2,}', text))
                    if "\n" in text:
                        first_line_text = text.split("\n")[0]
                        second_line_text = text.split("\n")[1]
                        # print(first_line_text)
                        # print(second_line_text)
                        if len(re.split(r'\s{2,}', first_line_text)) == 1:
                            agency_and_rating_text = re.split(r'\s{2,}', second_line_text)[1]
                            agency_matching_output = agency_matching(agency_and_rating_text)
                            remaining_text = agency_matching_output[0]
                            agency = agency_matching_output[1]
                            print("agency :", agency)


                            rating_matching_output = rating_matching(remaining_text)
                            remaining_text = rating_matching_output[0]
                            rating = rating_matching_output[1]
                            print("rating :", rating)

                            # print(re.split(r'\s{2,}', first_line_text)[0] + "\n" + re.split(r'\s{2,}', second_line_text)[0])
                            temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio"  : re.split(r'\s{2,}', first_line_text)[0] + "\n" + re.split(r'\s{2,}', second_line_text)[0],
                            # "Agency" : None,
                            # "Ratings" : re.split(r'\s{2,}', second_line_text)[1],
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : re.split(r'\s{2,}', second_line_text)[2],
                            "(%) Of Total AUM" : re.split(r'\s{2,}', second_line_text)[3],
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,
                            
                            }])
                        elif len(re.split(r'\s{2,}', second_line_text)) == 1:
                            # print(re.split(r'\s{2,}', first_line_text)[0] + "\n" + re.split(r'\s{2,}', second_line_text)[0])
                            agency_and_rating_text = re.split(r'\s{2,}', first_line_text)[1]
                            agency_matching_output = agency_matching(agency_and_rating_text)
                            remaining_text = agency_matching_output[0]
                            agency = agency_matching_output[1]
                            print("agency :", agency)


                            rating_matching_output = rating_matching(remaining_text)
                            remaining_text = rating_matching_output[0]
                            rating = rating_matching_output[1]
                            print("rating :", rating)

                            temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio"  : re.split(r'\s{2,}', first_line_text)[0] + "\n" + re.split(r'\s{2,}', second_line_text)[0],
                            # "Agency" : None,
                            # "Ratings" : re.split(r'\s{2,}', first_line_text)[1],
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : re.split(r'\s{2,}', first_line_text)[2],
                            "(%) Of Total AUM" : re.split(r'\s{2,}', first_line_text)[3],
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,
                            }])
                        correct_rating_df = pd.concat([correct_rating_df, temp_df])
                    else:
                        # print(text)
                        # print(len(re.split(r'\s{2,}', text)), re.split(r'\s{2,}', text))
                        splited_column_row = re.split(r'\s{2,}', text)
                        # print(re.split(r'\s{2,}', text)[1])
                        # rating_set.add(re.split(r'\s{2,}', text)[1])
                        # print(splited_column_row[1])
                        rating_set.add(splited_column_row[1])

                        agency_and_rating_text = splited_column_row[1]
                        agency_matching_output = agency_matching(agency_and_rating_text)
                        remaining_text = agency_matching_output[0]
                        agency = agency_matching_output[1]
                        print("agency :", agency)


                        rating_matching_output = rating_matching(remaining_text)
                        remaining_text = rating_matching_output[0]
                        rating = rating_matching_output[1]
                        print("rating :", rating)
                        temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio" : splited_column_row[0],
                            # "Agency" : None,
                            # "Ratings" : splited_column_row[1],
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : splited_column_row[2],
                            "(%) Of Total AUM" : splited_column_row[3],
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,

                        }])
                        correct_rating_df = pd.concat([correct_rating_df, temp_df])
                
                elif "SIP - If you had invested" in text:
                    print("SIP - If you had invested text appeared and skipped")
                elif len(normalized_text.split("\n")) ==1:
                    print("normalized_text :", normalized_text)


                    # portfolio = text.split("\n")[0]
                    agency_matching_output = agency_matching(normalized_text)
                    normalized_text = agency_matching_output[0]
                    agency = agency_matching_output[1]


                    rating_matching_output = rating_matching(normalized_text)
                    normalized_text = rating_matching_output[0]
                    rating = rating_matching_output[1]

                    AUM_matching_output = AUM_matching(normalized_text)
                    normalized_text = AUM_matching_output[0]
                    total_AUM = AUM_matching_output[1]

                    MV_matching_output = MV_matching(normalized_text)
                    normalized_text = MV_matching_output[0]
                    market_value = MV_matching_output[1]

                    portfolio = normalized_text.strip()

                    temp_df = pd.DataFrame([{
                        "Pan" : pan,
                        "ISIN No." : None,
                        "Mutual Fund" : mutual_fund,
                        "Fund Name" : filename.split("_page")[0].replace("_", " "),
                        "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                        "Portfolio" : portfolio,
                        "Agency" : agency,
                        "Ratings" : rating,
                        "Category" : None,
                        # "Market Value Rs Lakhs" : None,
                        "(%) Of Total AUM" : total_AUM,
                        "Factsheet Date" : None,
                        "URL" : None,
                        "Filename" : filename,

                    }])
                    correct_rating_df = pd.concat([correct_rating_df, temp_df])
                    print("remaining_text", normalized_text)
                    print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
                

                # else:
                    # temp_df = pd.DataFrame([{
                    #     "Pan" : pan,
                    #     "ISIN No." : None,
                    #     "Mutual Fund" : mutual_fund,
                    #     "Fund Name" : filename.split("_page")[0].replace("_", " "),
                    #     "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                    #     "Portfolio" : text,
                    #     "Agency" : None,
                    #     "Ratings" : None,
                    #     "Category" : None,
                    #     # "Market Value Rs Lakhs" : None,
                    #     "(%) Of Total AUM" : None,
                    #     "Factsheet Date" : None,
                    #     "URL" : None,
                    #     "Filename" : filename,

                    # }])
                    # correct_rating_df = pd.concat([correct_rating_df, temp_df])
                elif len(text.split("\n")) ==2:
                    if len(re.split(r'\s{2,}', text.split("\n")[0])) == 1:
                        # print("remaining_text_else")
                        # print(text)
                        portfolio = text.split("\n")[0]
                        agency_matching_output = agency_matching(text.split("\n")[1])
                        normalized_text = agency_matching_output[0]
                        agency = agency_matching_output[1]


                        agency_matching_output = rating_matching(normalized_text)
                        normalized_text = agency_matching_output[0]
                        rating = agency_matching_output[1]

                        AUM_matching_output = AUM_matching(normalized_text)
                        normalized_text = AUM_matching_output[0]
                        total_AUM = AUM_matching_output[1]

                        MV_matching_output = MV_matching(normalized_text)
                        normalized_text = MV_matching_output[0]
                        market_value = MV_matching_output[1]

                        portfolio = portfolio + "\n" + normalized_text

                        # print("agency", agency)
                        # print("rating", rating)
                        # print("total_AUM", total_AUM)
                        # print("market_value", market_value)
                        # print("portfolio :")
                        # print(portfolio)

                        temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio" : portfolio,
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : None,
                            "(%) Of Total AUM" : total_AUM,
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,

                        }])
                        correct_rating_df = pd.concat([correct_rating_df, temp_df])
                        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
                    else:
                        print("unresolved 2:", len(text.split("\n")),text)
                        line_1 = text.split("\n")[0]
                        agency_matching_output = agency_matching(line_1)
                        normalized_text = agency_matching_output[0]
                        agency = agency_matching_output[1]


                        agency_matching_output = rating_matching(normalized_text)
                        normalized_text = agency_matching_output[0]
                        rating = agency_matching_output[1]

                        AUM_matching_output = AUM_matching(normalized_text)
                        normalized_text = AUM_matching_output[0]
                        total_AUM = AUM_matching_output[1]

                        MV_matching_output = MV_matching(normalized_text)
                        normalized_text = MV_matching_output[0]
                        market_value = MV_matching_output[1]

                        portfolio = normalized_text.strip()

                        temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio" : portfolio,
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : None,
                            "(%) Of Total AUM" : total_AUM,
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,

                        }])
                        print(temp_df)
                        correct_rating_df = pd.concat([correct_rating_df, temp_df])
                        # print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")


                        line_2 = text.split("\n")[1]
                        agency_matching_output = agency_matching(line_2)
                        normalized_text = agency_matching_output[0]
                        agency = agency_matching_output[1]


                        agency_matching_output = rating_matching(normalized_text)
                        normalized_text = agency_matching_output[0]
                        rating = agency_matching_output[1]

                        AUM_matching_output = AUM_matching(normalized_text)
                        normalized_text = AUM_matching_output[0]
                        total_AUM = AUM_matching_output[1]

                        MV_matching_output = MV_matching(normalized_text)
                        normalized_text = MV_matching_output[0]
                        market_value = MV_matching_output[1]

                        portfolio = normalized_text.strip()

                        temp_df = pd.DataFrame([{
                            "Pan" : pan,
                            "ISIN No." : None,
                            "Mutual Fund" : mutual_fund,
                            "Fund Name" : filename.split("_page")[0].replace("_", " "),
                            "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                            "Portfolio" : portfolio,
                            "Agency" : agency,
                            "Ratings" : rating,
                            "Category" : None,
                            # "Market Value Rs Lakhs" : None,
                            "(%) Of Total AUM" : total_AUM,
                            "Factsheet Date" : None,
                            "URL" : None,
                            "Filename" : filename,

                        }])
                        correct_rating_df = pd.concat([correct_rating_df, temp_df])
                        print(temp_df)
                        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
                        
                else:
                    print("unresolved :", text)
                    temp_df = pd.DataFrame([{
                        "Pan" : pan,
                        "ISIN No." : None,
                        "Mutual Fund" : mutual_fund,
                        "Fund Name" : filename.split("_page")[0].replace("_", " "),
                        "Fund Type" : filename.split("_page")[0].replace("_", " ").split(' ', 1)[-1],
                        "Portfolio" : text,
                        "Agency" : None,
                        "Ratings" : None,
                        "Category" : None,
                        # "Market Value Rs Lakhs" : None,
                        "(%) Of Total AUM" : None,
                        "Factsheet Date" : None,
                        "URL" : None,
                        "Filename" : filename,

                    }])
                    correct_rating_df = pd.concat([correct_rating_df, temp_df])        


                print("--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

print("\nText extraction complete!")

def clean_decimal_values(MV):
    print("before", MV)
    try:
        if "." not in MV:
            MV = MV.replace(",", ".").replace(" ", ".")
            if "." not in MV:
                MV = MV[:-2] + "." + MV[-2:]
        elif "," in MV:
            # print("coming here")
            MV = MV.replace(",", "").replace(" ", "")
        elif " " in MV:
            MV = MV.replace(" ", "")
        print("after", MV)
        return MV
    except Exception as e:
        print(e)
        return None

# correct_rating_df['Market Value Rs Lakhs'] = correct_rating_df['Market Value Rs Lakhs'].apply(clean_decimal_values)
# correct_rating_df['% to NAV'] = correct_rating_df['% to NAV'].apply(clean_decimal_values)
# correct_rating_df.to_csv("correct_rating_df.csv", index=False)
# print("Processed Data Saved to csv")


def clean_portfolio(portfolio):
    if "GO!" in portfolio:
        portfolio = portfolio.replace("GO!", "GOI")
    if "GOI!" in portfolio:
        portfolio = portfolio.replace("GOI!", "GOI")
    if "Ses " in portfolio:
        portfolio = portfolio.replace("Ses ", "sgs ")
    return portfolio

correct_rating_df['Portfolio'] = correct_rating_df['Portfolio'].apply(clean_portfolio)

########################## Block to map isin number to fund name ##############################################
isin_df = pd.read_csv(r"D:\Nexensus_Projects\pdfEtraction\isin_Mastersheet.csv")
# Create a mapping dictionary
isin_map = dict(zip(isin_df['Fund Name'], isin_df['ISIN No.']))

# Add mapped ISIN to df2
correct_rating_df['ISIN No.'] = correct_rating_df['Fund Name'].map(isin_map)

# Save correct_rating_df to Excel file with separate sheets for each 'fund'
output_excel = "correct_rating_df_grouped_by_fund.xlsx"

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    for fund, group_df in correct_rating_df.groupby("Fund Name"):
        # Clean sheet name: max 31 characters, remove invalid characters
        sheet_name = str(fund)[:31]
        invalid_chars = r'[:\\/?*[\]]'
        sheet_name = re.sub(invalid_chars, "_", sheet_name)

        print(f"Writing sheet: {sheet_name}")
        group_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # group_df.drop(columns=["Fund Name"], errors="ignore").to_excel(writer, sheet_name=sheet_name, index=False)
        print("--------------------------------------------")

print(f"✅ Processed data saved to: {output_excel}")
    

---------------------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>--------------------------------------------------------------
---------------------------------------------------------->>>>>>>>>>>>>>>>>>>>>>>>--------------------------------------------------------------
screenshots_cropped_at_rectangles\Tata_Aggressive_Hybrid_Fund_page_1_table_1_below.png

Extracted Table Rows:
normalized_text : Debt Instruments
remaining_text Debt Instruments
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
normalized_text : Government Securities                                                                                     45646.69             11.07
Total AUM: 11.07

AUM_replaced_normalised_text : Government Securities                                  

PermissionError: [Errno 13] Permission denied: 'x.csv'

In [ ]:
###################### Narcotics india news scraping code ##################################################################
import requests
from bs4 import BeautifulSoup
import json
import time

newsSite = "Narcotics"
def write_to_txt_resolve(newData):
    file1 = r'D:\Nexensus_Projects\pdfEtraction\\data_{}.txt'.format(newsSite)
    text_file = open(file1, 'a')
    text_file.write(json.dumps(newData))
    text_file.write(",")
    text_file.write("\n")
    text_file.close()


header = {
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
        "Accept" : "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Cookie": '_ht_fp=WINHTNEW53801749811666; ppid=4192611473d3d243d10177b00806db7e5b5245baa725924aebdab8af2840be1a; _cb=sQzDtCaysICBMmSyz; _lc2_fpi=3423e49eb244--01jxmee7ttpcsc8wmn5qqdd9qt; _lc2_fpi_meta=%7B%22w%22%3A1749811666779%7D; _gcl_au=1.1.1027468645.1749811667; usercohortcitynew=Delhi; _ga=GA1.1.1226154917.1749811667; _cc_id=392a0ad35f69aeaa5a086268534a6b42; _scor_uid=70d644981d24478da4837753ae8a84da; _vwo_uuid_v2=D04DB616A21F3BC47D77E867A831DD341|d71b91edf2fc2e981fd978fba8cd07a1; _vwo_uuid=D04DB616A21F3BC47D77E867A831DD341; _vis_opt_s=1%7C; _clck=yaa34%7C2%7Cfy9%7C0%7C2045; FirstTimeVisitCookie=here; pubmatic-unifiedid=%7B%22TDID%22%3A%221f59fed1-dfce-46ab-85f6-0dd51badad26%22%2C%22TDID_LOOKUP%22%3A%22FALSE%22%2C%22TDID_CREATED_AT%22%3A%222025-08-12T11%3A40%3A32%22%7D; pubmatic-unifiedid_cst=zix7LPQsHA%3D%3D; ht-location=IN; Meta-Geo=IN--DL--NEWDELHI; HTMyOffer=myOfferHT; prContextualTags=BikeKeywords; _li_dcdm_c=.hindustantimes.com; panoramaId_expiry=1756807605986; _sp_ses.e8bf=*; cdpecho_country=IN; cdpecho_city=New Delhi; oneTap_viewed=true; g_oneTapShow=true; moe_uuid=c665ece9-08a0-44da-995e-d6b050ca68a5; gptScript=true; cdpCountrytrack=true; _chartbeat2=.1749811666504.1756722185974.0000000000000001.D36Tp9CmuNHICzXZcqW67ygT6JCu.1; _cb_svref=external; __gads=ID=d92bedca347cfd36:T=1749811667:RT=1756722185:S=ALNI_Ma0Bxtorm-9r3HmLPZNNqWEixp9tg; __gpi=UID=0000112ce9bfde98:T=1749811667:RT=1756722185:S=ALNI_MbQT5Y7BVauwWjOrlJTedqnrual7A; __eoi=ID=382fbc027fc564db:T=1749811667:RT=1756722185:S=AA-AfjYvQfj6L6Kn5Xa7nKZzDiuh; _sp_id.e8bf=ad71a14e-2ff0-4105-84ff-dd5201c2ce01.1749811667.30.1756722187.1755227920.974ea0ba-3d7b-4cc4-86bd-ed3d4a999375; _ga_9CYSK4EP0B=GS2.1.s1756721207$o31$g1$t1756722188$j60$l0$h0; moe_c_s=1; moe_u_d=%7B%22a%22%3A%5B%5D%2C%22s_old%22%3Afalse%2C%22d_id%22%3A%22c665ece9-08a0-44da-995e-d6b050ca68a5%22%2C%22d_added%22%3Atrue%7D; moe_s_n=%7B%22s_k%22%3A%2255c93a32-91bc-439a-92d9-6830d3b58a39%22%2C%22s_s_t%22%3A%222025-09-01T10%3A07%3A00.051Z%22%2C%22s_m_t%22%3A1800%2C%22c_i_t_t%22%3A%5B%5D%2C%22s_e_t%22%3A1756723997765%2C%22n_o_s%22%3A29%7D; cdp_page_referrer=; idleSliderShown=true',
        }

soup = BeautifulSoup(requests.get("https://narcoticsindia.nic.in/news.php").content, 'html.parser')
table_body = soup.find("table").find("tbody")
for row in table_body.find_all('tr'):
    title = row.find_all('td')[1].text.strip()
    link = row.find_all('td')[2].find('a')['href']
    print(requests.get(link, headers=header))
    newsData = {"heading": title, "link" : link, }
    print(newsData)
    write_to_txt_resolve(newsData)
    print("-------------------------------------------------------")

################################## SEBI news scraping code ################################################################
newsSite = "SEBI"

url = "https://www.sebi.gov.in/sebiweb/ajax/home/getnewslistinfo.jsp"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.sebi.gov.in/sebiweb/home/HomeAction.do?doListing=yes&sid=6&ssid=23&smid=0"
}

cookies = {
    "JSESSIONID": "C7F7C8E8722D1736110467AA0D643BD3"
}

# Base payload (except nextValue which we will change)
base_payload = {
    "next": "n",
    "search": "",
    "fromDate": "",
    "toDate": "",
    "fromYear": "",
    "toYear": "",
    "deptId": "-1",
    "sid": "6",
    "ssid": "23",
    "smid": "0",
    "ssidhidden": "23",
    "intmid": "-1",
    "sText": "Media & Notifications",
    "ssText": "Press Releases",
    "smText": "",
    # "doDirect": "3"
}

# Loop through pages
for page in range(1, 231):  # scrape first 231 pages
    time.sleep(1)
    payload = base_payload.copy()
    payload["nextValue"] = str(page)
    payload['doDirect'] = str(page)

    r = requests.post(url, headers=headers, cookies=cookies, data=payload)
    print(f"Page {page} -> {len(r.text)} characters")
    # print(r.content)
    soup = BeautifulSoup(r.content, 'html.parser')
    # print(soup.find('table', {'id' : 'sample_1'}).find_all('tr')[0])
    for row in soup.find('table', {'id' : 'sample_1'}).find_all('tr')[1:]:
        date = row.find_all('td')[0].text
        pr_no = row.find('th').text
        title  = row.find_all('td')[1].text
        link = row.find_all('td')[1].find('a')['href']
        print('date  :', date)
        print('pr_no :', pr_no)
        print('title :', title)
        print('href  :', link)
        newsData = {"heading": title, "pr_no": pr_no, "newsDate": date, "link" : link}
        print(newsData)
        write_to_txt_resolve(newsData)
    # parse r.text with BeautifulSoup or regex if it's HTML

##################################### CBI News scraping Code ##############################################################
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

newsSite = "CBI"

response = requests.get("https://cbi.gov.in/press-releases")
soup = BeautifulSoup(response.content, 'html.parser')

table_body = soup.find('table', {'id' : 'commonDataTable'}).find('tbody')

for row in table_body.find_all('tr'):
    # print(row)
    title = row.find_all('td')[1].text.replace("News Heading:", "")
    date = row.find_all('td')[2].text.replace("News Date:", "")
    link = row.find_all('td')[1].find("a")['href']
    try:
        # Create a session to persist cookies
        session = requests.Session()
        # Step 1: Set language to English
        lang_url = "https://cbi.gov.in/lang/en"
        session.get(lang_url, headers=headers, timeout=10)

        # Step 2: Now request the target page (will come in English)
        des_res = session.get(link, headers=headers, timeout=20)
        # des_res = requests.get(link,headers=headers, timeout=20)
        des_soup =  BeautifulSoup(des_res.content, 'html.parser')
        description = des_soup.find('div', 'press-txt').text
        # print("description :", description)
    except Exception as e:
        print('e :', e)
        time.sleep(20)
        # Create a session to persist cookies
        session = requests.Session()
        # Step 1: Set language to English
        lang_url = "https://cbi.gov.in/lang/en"
        session.get(lang_url, headers=headers, timeout=10)

        # Step 2: Now request the target page (will come in English)
        des_res = session.get(link, headers=headers, timeout=20)
        # des_res = requests.get(link,headers=headers, timeout=20)
        des_soup =  BeautifulSoup(des_res.content, 'html.parser')
        description = des_soup.find('div', 'press-txt').text
        # print("description :", description)
    print({"heading": title, "newsDate": date, "link" : link, "newsBody": description})
    newsData = {"heading": title, "newsDate": date, "link" : link, "newsBody": description}
    write_to_txt_resolve(newsData)
    time.sleep(2)
    print("------------")


##################################### NIA News scraping Code ##############################################################
newsSite = "NIA"

url = "https://nia.gov.in/press-releases.htm"

session = requests.Session()
resp = session.get(url)
soup = BeautifulSoup(resp.text, "html.parser")
for row in soup.find('tbody').find_all('tr'):
    s_no = row.find_all('td')[0].text.strip()
    newsDate = row.find_all('td')[1].text.strip()
    unit = row.find_all('td')[2].text.strip()
    category = row.find_all('td')[3].text.strip()
    heading = row.find_all('td')[4].text.strip()
    link = "https://nia.gov.in" +row.find_all('td')[5].find('a')['href']
    # print({"s_no": s_no, "heading": heading, "newsDate": newsDate, "link" : link})
    newsData = {"heading": heading, "newsDate": newsDate, "link" : link}
    print(newsData)
    write_to_txt_resolve(newsData)
    print("-----------------------------------------")

def get_hidden_fields(soup):
    fields = {}
    for field in ["__VIEWSTATE", "__VIEWSTATEGENERATOR", "__EVENTVALIDATION", '__VIEWSTATEENCRYPTED']:
        tag = soup.find("input", {"id": field})
        if tag:
            fields[field] = tag["value"]
    # print(fields.keys())
    return fields

for i in range(25):
    # Extract hidden fields from first page
    hidden = get_hidden_fields(soup)

    # Example: go to page 2
    payload = {
        **hidden,
        "__EVENTTARGET": "",  # from DevTools
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": "",
        # "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$dlCustomPager$ctl05$lnkbtnPaging" : 6,
        "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$ibtnMoveNext.x": 4,
        "ctl00$ContentPlaceHolder1$PressRlsList1$CustomPager1$ibtnMoveNext.y" : 18
    }

    resp2 = session.post(url, data=payload)
    soup = BeautifulSoup(resp2.text, "html.parser")
    # print(soup2)

    for row in soup.find('tbody').find_all('tr'):
        s_no = row.find_all('td')[0].text.strip()
        newsDate = row.find_all('td')[1].text.strip()
        unit = row.find_all('td')[2].text.strip()
        category = row.find_all('td')[3].text.strip()
        heading = row.find_all('td')[4].text.strip()
        link = "https://nia.gov.in" +row.find_all('td')[5].find('a')['href']
        # print({"s_no": s_no, "heading": heading, "newsDate": newsDate, "link" : link})
        newsData = {"heading": heading, "newsDate": newsDate, "link" : link}
        print(newsData)
        write_to_txt_resolve(newsData)
        print("-----------------------------------------")
        

<Response [200]>
{'heading': 'Only 4 Convictions in Over 13,000 Drug-Related Arrests in J&K Since 2019, Raising Serious Concerns', 'link': 'https://boldnewsonline.com/only-4-convictions-in-over-13000-drug-related-arrests-in-jk-since-2019-raising-serious-concerns/'}
-------------------------------------------------------
<Response [522]>
{'heading': '757 arrested in NDPS cases in Gujarat in 2024, says Centre in Lok Sabha; State wise figures here', 'link': 'https://deshgujarat.com/2025/02/04/757-arrested-in-ndps-cases-in-gujarat-in-2024-says-centre-in-lok-sabha-state-wise-figures-here/'}
-------------------------------------------------------
<Response [200]>
{'heading': 'The continuing drugs menace in Punjab', 'link': 'https://www.thehindu.com/opinion/op-ed/the-continuing-drugs-menace-in-punjab/article69179452.ece'}
-------------------------------------------------------
<Response [200]>
{'heading': 'Bhullar suspends two transport employees for ‘peddling drugs', 'link': 'https://www.hin